# FMCG Profit Prediction

End-to-end machine learning regression project predicting order-level  across 18,240 FMCG transactions (2023–2025). Covers EDA, feature engineering, preprocessing, multi-model comparison, and evaluation.

## 1. Exploratory Data Analysis

We start by understanding the structure and quality of the data. No transformations happen here — just observation. All plots use seaborn's default theme.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme()
%matplotlib inline

In [ ]:
df = pd.read_csv("../data/fmcg_sales_marketing_profitability_2023_2025.csv")
print(f"Shape: {df.shape}")
print()
print(df.dtypes)

### 1.1 Missing Values

Check both standard nulls and blank strings stored as objects, which are a common hidden null pattern in exported CSV data.

In [ ]:
print("Null counts:")
print(df.isnull().sum())
print()
# Hidden nulls: blank strings in object columns
object_cols = df.select_dtypes(include="object").columns
blank_counts = (df[object_cols] == "").sum()
print("Blank string counts in object columns:")
print(blank_counts)

### 1.2 Statistical Summary

Numeric columns only. We pay attention to the spread of  — wide standard deviation relative to the mean signals high variance and a meaningful prediction challenge.

In [ ]:
display(df.describe().round(2))

### 1.3 Target Distribution

 is right-skewed with a long positive tail and a small left tail of unprofitable orders. We plot both a histogram and a box plot to capture shape and outliers.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df["Profit_USD"], bins=60, kde=True, ax=axes[0])
axes[0].set_title("Profit_USD Distribution")
axes[0].set_xlabel("Profit (USD)")

sns.boxplot(x=df["Profit_USD"], ax=axes[1])
axes[1].set_title("Profit_USD Box Plot")
axes[1].set_xlabel("Profit (USD)")

plt.tight_layout()
plt.show()

print(f"Negative profit orders: {(df['Profit_USD'] < 0).sum()} ({(df['Profit_USD'] < 0).mean():.1%})")

### 1.4 Profit by Categorical Features

We want to understand which segments drive profitability. These box plots show median profit and spread within each group.

In [ ]:
cat_cols = ["Product_Category", "Region", "Sales_Channel", "Customer_Type", "Promotion_Type", "Brand"]

fig, axes = plt.subplots(3, 2, figsize=(14, 14))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    order = df.groupby(col)["Profit_USD"].median().sort_values(ascending=False).index
    sns.boxplot(data=df, x=col, y="Profit_USD", order=order, ax=axes[i])
    axes[i].set_title(f"Profit by {col}")
    axes[i].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

### 1.5 Numeric Feature Distributions

We look at how continuous inputs are distributed — this tells us what preprocessing they need (scaling, imputation) and whether any relationships with the target are visible.

In [ ]:
num_cols = ["Units_Sold", "Unit_Price_USD", "Discount_Pct", "Marketing_Spend_USD"]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.histplot(df[col], bins=40, kde=True, ax=axes[i])
    axes[i].set_title(f"{col} Distribution")

plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots against Profit_USD
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    # sample to keep plot readable
    sample = df.sample(n=2000, random_state=42)
    sns.scatterplot(data=sample, x=col, y="Profit_USD", alpha=0.4, ax=axes[i])
    axes[i].set_title(f"{col} vs Profit_USD")

plt.tight_layout()
plt.show()

### 1.6 Correlation Heatmap

Correlation between numeric features and the target. This drives which raw features carry signal and which engineered features we should create.

In [ ]:
# Include only non-leakage numeric columns for the correlation view
corr_cols = ["Units_Sold", "Unit_Price_USD", "Discount_Pct", "Marketing_Spend_USD",
             "Year", "Month", "Profit_USD"]
corr = df[corr_cols].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Correlation Heatmap (Non-Leakage Features)")
plt.tight_layout()
plt.show()

### 1.7 Profit by Year and Quarter

Time patterns can reveal seasonal trends or year-over-year growth that a model should capture.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.boxplot(data=df, x="Year", y="Profit_USD", ax=axes[0])
axes[0].set_title("Profit by Year")

sns.boxplot(data=df, x="Quarter", y="Profit_USD",
            order=["Q1","Q2","Q3","Q4"], ax=axes[1])
axes[1].set_title("Profit by Quarter")

plt.tight_layout()
plt.show()

## 2. Feature Engineering

Based on EDA, we create features that capture business logic and interaction effects. We also drop columns that would leak the target or are pure identifiers.

**Dropped columns:**
- , ,  — identifiers / redundant with time columns
- , , ,  — direct arithmetic components of ; including them would make the target trivially reconstructable
-  — derived from , causes direct leakage

**New features:**
- : unit price after discount — captures the real price a customer pays
- : units × effective price — a proxy for top-line revenue without using the official (post-cost) revenue columns
- : spend per unit sold — measures marketing efficiency
- : binary flag for any active promotion — condenses the promotion type into a signal
- : binary flag for the holiday quarter — captures the seasonal lift visible in EDA

In [ ]:
# Work on a copy to preserve the raw dataframe
df_fe = df.copy()

# New features
df_fe["Effective_Unit_Price"] = df_fe["Unit_Price_USD"] * (1 - df_fe["Discount_Pct"] / 100)
df_fe["Revenue_Estimate"]     = df_fe["Units_Sold"] * df_fe["Effective_Unit_Price"]
df_fe["Marketing_Per_Unit"]   = df_fe["Marketing_Spend_USD"] / df_fe["Units_Sold"]
df_fe["Is_Promoted"]          = (df_fe["Promotion_Type"] != "No Promo").astype(int)
df_fe["Is_Q4"]                = (df_fe["Quarter"] == "Q4").astype(int)

# Drop leakage and identifier columns
leakage_cols = [
    "Order_ID", "SKU", "Order_Date",
    "Gross_Sales_USD", "Net_Revenue_USD",
    "COGS_USD", "Logistics_Cost_USD",
    "Profit_Margin_Pct"
]
df_fe = df_fe.drop(columns=leakage_cols)

print(f"Shape after feature engineering: {df_fe.shape}")
print(f"
Columns remaining:
{list(df_fe.columns)}")
print(f"
New feature samples:")
display(df_fe[["Effective_Unit_Price","Revenue_Estimate","Marketing_Per_Unit","Is_Promoted","Is_Q4"]].describe().round(2))

## 3. Train / Test Split

We split 80/20 before any preprocessing. The test set is held completely aside until the final evaluation step. This is a regression problem so no stratification is needed — we use  with a fixed random state for reproducibility.

In [ ]:
from sklearn.model_selection import train_test_split

target = "Profit_USD"
X = df_fe.drop(columns=[target])
y = df_fe[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f"Train: X={X_train.shape}, y={y_train.shape}")
print(f"Test:  X={X_test.shape}, y={y_test.shape}")
print(f"
Target stats — train: mean={y_train.mean():.2f}, std={y_train.std():.2f}")
print(f"Target stats — test:  mean={y_test.mean():.2f}, std={y_test.std():.2f}")

## 4. Preprocessing Pipeline

We build a  inside a  that applies different steps to numeric and categorical columns:

- **Numeric**: median imputation → standard scaling
- **Categorical**: most-frequent imputation → ordinal encoding with 

Critically,  is called only on the training set. The test set only sees . This ensures that the scaler means/variances and the encoder category maps are learned exclusively from training data, preventing any information from the test set from influencing preprocessing — which would be data leakage.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

# Identify numeric and categorical columns in X_train
numeric_cols = X_train.select_dtypes(include=["int64","float64"]).columns.tolist()
categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()

print(f"Numeric columns ({len(numeric_cols)}): {numeric_cols}")
print(f"
Categorical columns ({len(categorical_cols)}): {categorical_cols}")

In [ ]:
# Build the preprocessor
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1
    ))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline,  numeric_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

# Fit on train only — transform both
# This is the crucial step that prevents data leakage
X_train_arr = preprocessor.fit_transform(X_train)
X_test_arr  = preprocessor.transform(X_test)

print(f"X_train processed shape: {X_train_arr.shape}")
print(f"X_test  processed shape: {X_test_arr.shape}")

## 5. Multiple Model Comparison

We train nine regressors on the preprocessed training data and compare them on the test set. Primary metric is R² (proportion of variance explained). Secondary metrics are RMSE (penalises large errors) and MAE (more robust to outliers). We also run 5-fold cross-validation scored on R² to check that test performance is not an artefact of the particular split.

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings("ignore")

models = {
    "Linear Regression":    LinearRegression(),
    "Ridge":                Ridge(alpha=1.0),
    "Lasso":                Lasso(alpha=0.1),
    "Decision Tree":        DecisionTreeRegressor(random_state=42),
    "Random Forest":        RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "Gradient Boosting":    GradientBoostingRegressor(n_estimators=100, random_state=42),
    "AdaBoost":             AdaBoostRegressor(n_estimators=100, random_state=42),
    "SVR":                  SVR(kernel="rbf"),
    "KNN":                  KNeighborsRegressor(n_neighbors=5),
}

In [ ]:
results = []

for name, model in models.items():
    model.fit(X_train_arr, y_train)
    preds = model.predict(X_test_arr)

    r2   = r2_score(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae  = mean_absolute_error(y_test, preds)

    cv_scores = cross_val_score(
        model, X_train_arr, y_train,
        cv=5, scoring="r2", n_jobs=-1
    )

    results.append({
        "Model":   name,
        "R2":      round(r2, 4),
        "RMSE":    round(rmse, 2),
        "MAE":     round(mae, 2),
        "CV Mean": round(cv_scores.mean(), 4),
        "CV Std":  round(cv_scores.std(), 4),
    })
    print(f"{name:<25} R2={r2:.4f}  RMSE={rmse:.2f}  MAE={mae:.2f}  CV={cv_scores.mean():.4f}±{cv_scores.std():.4f}")

results_df = pd.DataFrame(results).sort_values("R2", ascending=False).reset_index(drop=True)
print("
--- Comparison Table (sorted by R²) ---")
display(results_df)

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]
print(f"Best model: {best_model_name}  (R² = {results_df.iloc[0]['R2']})")

## 6. Evaluation

We produce a comprehensive set of diagnostic plots to understand where each model performs well and where it falls short.

### 6.1 Model Comparison Bar Chart

In [ ]:
metrics = ["R2", "RMSE", "MAE"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, metric in zip(axes, metrics):
    ascending = metric != "R2"   # lower is better for RMSE and MAE
    plot_df = results_df.sort_values(metric, ascending=ascending)
    sns.barplot(data=plot_df, x=metric, y="Model", ax=ax)
    ax.set_title(f"{metric} by Model")
    ax.set_xlabel(metric)
    ax.set_ylabel("")

plt.suptitle("Model Comparison", y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

### 6.2 Actual vs Predicted — Top 4 Models

The ideal model produces points along the diagonal. Scatter above or below reveals systematic bias.

In [ ]:
top4_names = results_df.head(4)["Model"].tolist()

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, name in enumerate(top4_names):
    model = models[name]
    preds = model.predict(X_test_arr)
    sns.scatterplot(x=y_test, y=preds, alpha=0.3, ax=axes[i])
    lims = [min(y_test.min(), preds.min()), max(y_test.max(), preds.max())]
    axes[i].plot(lims, lims, "r--", linewidth=1, label="Perfect fit")
    axes[i].set_xlabel("Actual Profit_USD")
    axes[i].set_ylabel("Predicted Profit_USD")
    axes[i].set_title(name)
    axes[i].legend()

plt.suptitle("Actual vs Predicted — Top 4 Models", y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

### 6.3 Residual Plots — Top 4 Models

Residuals (actual − predicted) should be centred around zero with no pattern. Fans or curves reveal heteroscedasticity or non-linearity the model is missing.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, name in enumerate(top4_names):
    model = models[name]
    preds = model.predict(X_test_arr)
    residuals = y_test.values - preds
    sns.scatterplot(x=preds, y=residuals, alpha=0.3, ax=axes[i])
    axes[i].axhline(0, color="red", linestyle="--", linewidth=1)
    axes[i].set_xlabel("Predicted Profit_USD")
    axes[i].set_ylabel("Residual")
    axes[i].set_title(f"Residuals — {name}")

plt.suptitle("Residual Plots — Top 4 Models", y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

### 6.4 Feature Importance — Random Forest

Random Forest provides impurity-based importance scores. These tell us which features the model relied on most across all trees.

In [ ]:
rf_model = models["Random Forest"]
feature_names = numeric_cols + categorical_cols
importances = pd.Series(rf_model.feature_importances_, index=feature_names).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=importances.values, y=importances.index)
plt.title("Random Forest Feature Importances")
plt.xlabel("Mean Decrease in Impurity")
plt.tight_layout()
plt.show()

### 6.5 Best Model — Detailed Report

In [ ]:
best_preds = best_model.predict(X_test_arr)
best_residuals = y_test.values - best_preds

print(f"=== Best Model: {best_model_name} ===")
print(f"R²:   {r2_score(y_test, best_preds):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, best_preds)):.2f}")
print(f"MAE:  {mean_absolute_error(y_test, best_preds):.2f}")
print()
print("Residual Summary:")
print(pd.Series(best_residuals).describe().round(2))

# Residual distribution
plt.figure(figsize=(8, 4))
sns.histplot(best_residuals, bins=50, kde=True)
plt.axvline(0, color="red", linestyle="--", linewidth=1)
plt.title(f"Residual Distribution — {best_model_name}")
plt.xlabel("Residual (Actual − Predicted)")
plt.tight_layout()
plt.show()